# Play dice with more than one agent

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from dice.agents import Agent, SmartAgent, StopAtNDiceAgent, determine_available_actions
from dice.game import FillOrBustEnvironment

# Utility functions

In [3]:
def plot_training_metric(
    agent: Agent,
    metric: str = "num_turns_to_win",
    downsample: bool = True,
) -> None:
    # Format the data
    data = getattr(agent, metric)
    if downsample:
        data = data[:: len(data) // min(2500, len(data))]
    source = (
        pd.DataFrame(data, columns=[metric])
        .reset_index()
        .rename(columns={"index": "episode"})
    )
    print(f"Median `{metric}` after training: {source.tail(50)[metric].median()}")

    # Create the chart
    padding = int(0.1 * len(source))
    line = (
        alt.Chart(source)
        .mark_line(color="red", size=3)
        .transform_window(
            rolling_mean=f"median({metric})",
            frame=[-padding, padding],
        )
        .encode(x="episode", y="rolling_mean:Q")
    )
    points = (
        alt.Chart(source)
        .mark_point(opacity=0.2, color="orange")
        .encode(x="episode", y=alt.Y(f"{metric}:Q").title(metric))
    )

    chart = (points + line).properties(width=400, height=250)
    chart.show()

# Train an agent

In [4]:
n_episodes = 10_000
start_learning_rate = 0.001
lr_decay = start_learning_rate / (n_episodes // 2)
final_lr = 0.0001
start_epsilon = 1.0
epsilon_decay = start_epsilon / (n_episodes // 2)
final_epsilon = 0.1
discount_factor = 0.8

kwargs = dict(
    initial_learning_rate=start_learning_rate,
    learning_rate_decay=lr_decay,
    final_learning_rate=final_lr,
    initial_epsilon=start_epsilon,
    epsilon_decay=epsilon_decay,
    final_epsilon=final_epsilon,
    discount_factor=discount_factor,
)

In [5]:
agent1 = SmartAgent("agent1", **kwargs)
# agent2 = SmartAgent("agent2", **kwargs)

# agent1 = Agent("agent1")
agent2 = Agent("agent2")
# agent3 = Agent("agent3")

agents: list[Agent] = [agent1]

env = FillOrBustEnvironment()
for episode in tqdm(range(n_episodes)):
    obs, _ = env.reset_for_new_game()
    done, turn_is_over = False, False
    highest_score, winner = 0, None

    # play one episode
    while not done:
        for agent in agents:
            while not turn_is_over:
                action = agent.get_action(obs)
                next_obs, reward, terminated, truncated, turn_is_over, info = env.step(
                    agent, action
                )

                agent.update(
                    obs, action, reward, terminated, turn_is_over, next_obs, info
                )

                done = terminated or truncated
                obs = next_obs
            turn_is_over = False
            if done:
                break

    for agent in agents:
        if agent.game_score > highest_score:
            highest_score = agent.game_score
            winner = agent
        agent.turn_count = 0
        agent.game_score = 0
        if isinstance(agent, SmartAgent):
            agent.decay_epsilon()
            agent.decay_learning_rate()

    for agent in agents:
        if agent == winner:
            agent.wins += 1
            break

100%|██████████| 10000/10000 [00:19<00:00, 515.92it/s]


In [ ]:
print(f"Agent 1 wins: {agent1.wins}")
print(f"Agent 2 wins: {agent2.wins}")
100 * agent1.wins / (agent1.wins + agent2.wins)

In [ ]:
plot_training_metric(agent, "rewards")

In [ ]:
s = pd.Series(agent1.num_turns)

In [ ]:
(-s.diff()[s.diff() < 0]).plot(style="k.-", alpha=0.3, grid=True)
plt.show()

In [ ]:
pd.Series(agent1.rewards).value_counts()

In [ ]:
pd.Series(agent2.rewards).value_counts()

In [ ]:
agent1_scores_per_turn = pd.Series(agent1.game_scores, index=agent1.num_turns)
agent2_scores_per_turn = pd.Series(agent2.game_scores, index=agent2.num_turns)

In [ ]:
num_steps = 5000
agent1_scores_per_turn.head(num_steps).plot(style="ko-", grid=True, label=agent1.name)
agent2_scores_per_turn.head(num_steps).plot(style="ro-", grid=True, label=agent2.name, alpha=0.3)
plt.ylabel("Game Score")
plt.legend()
plt.show()

In [ ]:
def sort_q_values(q):
    return {
        (
            obs.hold_score,
            obs.num_dice_remaining,
            obs.dice,
            f"{obs.card.category}_{obs.card.bonus}" if obs.card else "",
        ): {
            key[1]: q[obs][key]
            for key in sorted(q[obs].keys(), key=q[obs].get, reverse=True)
        }
        for obs in sorted(
            q.keys(),
            key=lambda o: sum(q[o].values()),
            # key=lambda o: o.hold_score,
            reverse=True,
        )
        if any(action[0] == "inherit" for action in q[obs].keys())
    }

In [ ]:
q = sort_q_values(agent1.q_values)

In [ ]:
q